# 모듈 1: 블랙파우더 성분 데이터 전처리 - EDA 및 검증

`src/pipeline.py`의 각 단계를 순서대로 호출해 원본 데이터를 정제하고, 전/후 비교 시각화와 파생변수·원료등급 기여도(ablation test)를 확인한다.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from pathlib import Path

from src.preprocessing import load_raw, clean_schema
from src.imputation import flag_outliers, impute_missing
from src.features import engineer_features
from src.validation import (
    check_ranges,
    plot_missing_before_after,
    plot_distributions,
    plot_correlation_heatmap,
    run_ablation_test,
    plot_ablation_results,
    plot_grade_distribution,
)
from src.config import COMPOSITION_COLUMNS, PROCESS_COLUMNS, TARGET_COLUMNS

RAW_PATH = Path('../data/raw/영풍그룹_금속회수율최적화_RawMaterialData_학생배포용.xlsx')
FIGURES_DIR = Path('../reports/figures')

## 1. 원본 로드 및 정제

In [ ]:
raw_df = load_raw(RAW_PATH)
clean_df = clean_schema(raw_df)
clean_df.shape

### 1.1 완전 중복 배치 제거

`배치ID` 기준 중복 4건(8행)은 분석일자·원료등급·회수율까지 완전히 동일한 복사본으로 확인됐다 — 서로 다른 두 배치에 ID가 우연히 겹친 게 아니라 같은 레코드가 두 번 입력된 것이다. `clean_schema`가 남긴 `is_duplicate_batch` 플래그를 근거로 첫 occurrence만 남기고 제거한다 — 그대로 두면 교차검증에서 동일 데이터 포인트가 train/test 양쪽에 걸쳐 들어가 성능 지표가 부풀려질 수 있다. 남은 행의 `is_duplicate_batch=True`는 "원래 중복이 있었고 하나는 제거됐다"는 감사 기록으로 유지된다 (`src/pipeline.py`의 `run()`과 동일한 순서).

In [ ]:
clean_df = clean_df.drop_duplicates(subset='배치ID', keep='first')
print(f'중복 제거 후: {len(clean_df)}행')

In [ ]:
outlier_flagged_df = flag_outliers(clean_df)
imputed_df = impute_missing(outlier_flagged_df)
final_df = engineer_features(imputed_df)
final_df.head()

## 2. 결측치 처리 전/후 비교

In [ ]:
plot_missing_before_after(outlier_flagged_df, imputed_df, FIGURES_DIR)

## 3. 이상치 제거 전/후 분포

In [ ]:
plot_distributions(raw_df, ['Ni_함량_pct', '황산투입량_L', '침출온도_C'], FIGURES_DIR / 'before')
plot_distributions(final_df, ['Ni_함량_pct', '황산투입량_L', '침출온도_C'], FIGURES_DIR / 'after')

## 4. 상관관계 히트맵

In [ ]:
feature_columns = COMPOSITION_COLUMNS + PROCESS_COLUMNS + [
    'acid_to_ore_ratio', 'leach_intensity', 'total_impurity_pct', 'target_purity_ratio',
    'Fe_Ni_ratio', 'Cu_Ni_ratio', 'Al_Ni_ratio'
]
plot_correlation_heatmap(final_df, feature_columns, TARGET_COLUMNS, FIGURES_DIR)

조성 변수 중 목표금속(Ni_함량_pct, Co_함량_pct, Li_함량_pct)은 회수율과 양의 상관(+0.30~+0.49), 불순물(Cu/Fe/Al_불순물_pct)과 Mn_함량_pct는 음의 상관(-0.26~-0.57)을 보여 — 원료 안에 목표금속이 많을수록, 불순물이 많을수록 회수율에 미치는 방향이 §6 특성공학에서 가정한 도메인 지식(경쟁 반응 가설)과 일치한다. 공정조건(침출온도_C, 침출시간_hr)도 양의 상관(+0.27~+0.48)을 보여 반응속도론적 기대와 부합한다.

특히 파생변수(acid_to_ore_ratio, leach_intensity, total_impurity_pct, target_purity_ratio)는 대체로 그 구성요소인 원본 변수 개별 상관보다 더 강하다 — 예: leach_intensity(Ni 0.51)는 구성요소인 침출온도_C(0.48)·침출시간_hr(0.36) 개별 상관보다 높고, total_impurity_pct(Co -0.60)도 Cu/Fe/Al_불순물_pct 개별 상관(-0.53~-0.57)보다 더 강하다. 이는 §6 ablation test에서 파생변수가 R²를 크게 끌어올린 결과(Ni Validation R² 0.48→0.75)와 방향이 일치한다.

반면 Fe_Ni_ratio·Cu_Ni_ratio·Al_Ni_ratio도 -0.41~-0.54 수준의 꽤 강한 단순 상관을 보이는데, §6.1 ablation test에서는 모델 R²에 전혀 기여하지 못했다(§6.1 결론 참고) — 변수 하나와 타겟 사이의 단순 상관과, 다른 변수들과 함께 썼을 때의 모델 기여도는 서로 다른 질문이라는 걸 보여주는 사례다.

## 5. 정합성 검증

In [ ]:
violations = check_ranges(final_df)
print('정상범위 위반:', violations if violations else '없음')

## 6. 파생변수 및 원료등급 기여도 검증 (Ablation Test)

In [ ]:
base_features = COMPOSITION_COLUMNS + PROCESS_COLUMNS
# analysis_month는 회수율 예측 후보 변수가 아니라 계절성/설비 드리프트 모니터링용 캘린더 파생변수이므로, 특성 기여도 비교(히트맵·ablation test) 대상에서 제외한다.
derived_features = ['acid_to_ore_ratio', 'leach_intensity', 'total_impurity_pct', 'target_purity_ratio']

feature_sets = {
    'base_only': base_features,
    'base_plus_grade': base_features + ['원료등급'],
    'base_plus_derived': base_features + derived_features,
    'all_features': base_features + ['원료등급'] + derived_features,
}

results = run_ablation_test(outlier_flagged_df, feature_sets, TARGET_COLUMNS)
pd.DataFrame(results).T

In [ ]:
plot_ablation_results(results, FIGURES_DIR)

### 6.1 개별 불순물 비율(Fe/Ni, Cu/Ni, Al/Ni) 추가 검증

`total_impurity_pct`는 Cu/Fe/Al 불순물을 동일 가중치로 합산해, 어느 불순물이 실제로 회수율 저하를 유발하는지 정보를 가린다. 위 §6 ablation과 동일한 4-케이스 구조로, 비율 변수가 (a) 단독으로도 신호가 있는지, (b) 이미 검증된 최선 조합(`base_plus_derived`) 위에 추가로 기여하는지를 함께 검증한다.

In [ ]:
impurity_ratio_features = ['Fe_Ni_ratio', 'Cu_Ni_ratio', 'Al_Ni_ratio']

ratio_feature_sets = {
    'base_only': base_features,
    'base_plus_ratios': base_features + impurity_ratio_features,
    'base_plus_derived': base_features + derived_features,
    'base_plus_derived_plus_ratios': base_features + derived_features + impurity_ratio_features,
}

ratio_results = run_ablation_test(outlier_flagged_df, ratio_feature_sets, TARGET_COLUMNS)
pd.DataFrame(ratio_results).T

In [ ]:
plot_ablation_results(ratio_results, FIGURES_DIR / 'ratios')

**결론**: 실측 결과는 다음과 같다 (Validation/Test 둘 다 함께 봐야 하는 이유는 바로 아래에서 드러난다).

- `base_only` → `base_plus_ratios`: Ni Validation R² 0.4836→0.4695(차이 -0.0141), Test R² 0.5157→0.5186(차이 +0.0029) — **Validation은 떨어지고 Test는 오히려 오른다.** 방향 자체가 반대라는 게 "진짜 효과가 아니라 분할에 따라 흔들리는 노이즈"라는 가장 명확한 증거다. Co도 동일(Validation -0.0194, Test +0.0034).
- `base_plus_derived` → `base_plus_derived_plus_ratios`: Ni Validation 0.7505→0.7428(-0.0077), Test 0.7379→0.7230(-0.0149) — 이번엔 두 지표가 같은 방향(하락)이지만 폭이 0.01~0.015 수준으로 작다. Co도 두 지표 모두 -0.002 수준.

비교 기준으로, `base_only` → `base_plus_derived`(파생변수 4종 추가)는 Validation·Test 양쪽 모두에서 Ni +0.18~0.27, Co +0.18~0.20 수준으로 뚜렷하게, 같은 방향으로 상승한다 — 방향이 일치하고 폭도 불순물비율 효과의 10배 이상이라, 파생변수의 기여는 질적으로 다른 수준이다. 상관관계 히트맵에서는 Fe_Ni_ratio·Cu_Ni_ratio·Al_Ni_ratio가 타겟과 -0.4~-0.5 수준의(결코 약하지 않은) 단순 상관을 보였는데도 모델 R²에는 실질적으로 기여하지 못했다는 점이 흥미롭다 — RandomForest는 이미 `Cu/Fe/Al_불순물_pct`와 `Ni_함량_pct` 원본 컬럼에 직접 접근할 수 있어서, 비율을 미리 계산해주지 않아도 트리 분기로 비슷한 경계를 스스로 학습할 수 있기 때문으로 보인다(이는 비율 항이 명시적으로 필요한 선형모델과의 차이점이다). 따라서 이 세 비율 변수는 `engineer_features`에는 유지하되(해석용 참고 지표로서의 가치는 있을 수 있음), 최종 모델 입력 특성 세트에는 포함하지 않기로 결론짓는다 — "단순 상관계수가 높아 보인다"는 것과 "모델에 실제로 기여한다"는 것은 다른 질문이며, 후자는 반드시 (Validation과 Test를 같이 보면서) ablation test로 검증해야 한다는 것을 보여주는 사례다.

### 6.2 Data Leakage를 피하기 위한 평가 설계

모델 평가가 공정하려면 Test가 "진짜 처음 보는 데이터"로 남아있어야 한다. 이를 위해 `run_ablation_test`는 두 가지를 지킨다.

**1) 시간순 고정 분할 (무작위 섞기 없음)**: 배치는 2025년 1~10월에 걸쳐 순차적으로 들어온 데이터다. 이 모델이 실제로 쓰이는 상황(과거 배치로 학습해 앞으로 들어올 미래 배치의 회수율을 예측)을 그대로 재현하려면, 무작위로 섞어서 평가하면 안 된다 — 그러면 미래 배치로 학습해서 과거 배치를 맞히는 비현실적인 경우가 섞여 들어가고, 설비 캘리브레이션 드리프트 같은 시간에 따른 변화도 못 잡아낸다. 그래서 배치ID 순번(분석일자 순서의 근사)으로 정렬한 뒤 앞 70%를 Train, 다음 15%를 Validation, 마지막 15%를 Test로 고정 분할한다(`_temporal_split`, 294/63/63행).

**2) 결측치 대치 통계는 Train에서만 계산**: 분할 이후 Validation·Test 쪽 결측치를 채울 때도 중앙값·회귀계수는 Train 데이터로만 계산해서 적용한다(`_fold_safe_prepare`). 그래야 Validation·Test 행의 값이 자기 자신의 대치값 계산에 섞여 들어가지 않는다. 초기 구현은 이 순서를 지키지 않고 전체 420행 기준 통계로 먼저 대치한 뒤 무작위 5-fold로 나눴는데, 영향을 실측해보니 차이는 0.0014 이하(fold 표준편차 0.02~0.06 대비 노이즈 수준)로 작았지만, 작다고 구조적 문제가 없어지는 건 아니라서 두 가지 다 구조적으로 고쳐뒀다.

Validation은 피처 조합 비교(§6, §6.1의 목적)에 쓰고, Test는 최종 확인용으로 함께 보고한다. 이렇게 "미래를 미리 들여다보는" 흔한 실수 두 가지(무작위 분할, 분할 전 통계 계산)를 모두 피한다.

## 7. 심층 인터뷰 답변

**1) 실제 공정 데이터 품질 문제를 어떻게 식별·처리했는가?**

데이터사전에 정의된 정상 범위를 벗어나는 값(`Ni_함량_pct=999.0`, `침출온도_C=-5.0`, `황산투입량_L=15000`)을 물리적으로 불가능한 이상치로 식별해 `flag_outliers`에서 NaN으로 치환했다. 이후 남은 결측(조성 6~9%, 공정조건)은 — 원료등급과의 카이제곱 독립성 검정(p>0.05) 및 컬럼 간 결측 동시발생 상관관계(~0) 확인 결과 특정 조건에 치우친 체계적 패턴이 발견되지 않아 — 변수 성격에 따라 원료등급별 중앙값, 회귀 기반, 배치ID 순서 기반 보간으로 구분 처리했다 (설계 문서 §4).

**2) 특성공학 과정에서 도메인 지식을 어떻게 활용했는가?**

산-광석비, 침출강도(온도×시간), 총불순물, 목표금속 순도비는 모두 습식제련 침출 공정의 화학양론·반응속도론적 지식에서 도출했다 (설계 문서 §6). 위 ablation test로 파생변수·원료등급 포함 시 R²/RMSE가 실제로 개선되는지 정량 확인했다.

**3) 데이터 불균형 문제가 있었다면 어떻게 해결했는가?**

전체 420행(중복 배치 제거 후) 기준 `원료등급`(A/B/C) 분포는 B 38.6%, A 35.2%, C 26.2%로(아래 셀 실행 결과), 가장 적은 클래스(C)도 약 26%를 차지해 특정 클래스가 극단적으로 희소한 심각한 불균형은 아니라고 판단했다. 설계 문서 §9에서는 구현 단계의 실제 분포를 보고 불균형 여부와 계층적 샘플링(stratified split) 적용 여부를 결정하기로 했는데, 실측 결과 클래스 간 비율 차이가 26~39% 수준으로 완만해 SMOTE·계층적 샘플링·클래스 가중치 등 별도의 불균형 보정은 적용하지 않기로 결론지었다. 다만 위 ablation test 결과(시간순 Train/Validation/Test 분할, §6.2 참고)를 보면 `base_only`(Ni Validation R²=0.4836, Test R²=0.5157) 대비 `base_plus_grade`(Ni Validation R²=0.4618, Test R²=0.5152)는 Validation·Test가 서로 다른 신호를 준다(Validation은 0.02 하락, Test는 거의 그대로) — Co는 아예 방향이 반대다(Validation -0.0023, Test +0.0110). 두 독립적인 홀드아웃에서 방향조차 일관되지 않는다는 건 실제 효과가 아니라 분할에 따라 흔들리는 노이즈라는 뜻이다 — `원료등급`을 모델 입력 특성으로 그대로 추가하는 것 자체는 회귀 성능에 안정적인 영향을 주지 않는다. 그럼에도 `원료등급`은 Task 5의 등급별 중앙값 보간(grade-based median imputation)에서 그룹핑 키로 쓰이므로 컬럼 자체는 계속 유지한다. 반면 산-광석비·침출강도·총불순물·목표금속 순도비 등 파생변수는 `base_plus_derived`에서 R²를 Ni Validation 0.4836→0.7505·Test 0.5157→0.7379, Co Validation 0.5344→0.7168·Test 0.5970→0.7759 수준으로 Validation·Test 양쪽 모두 뚜렷하고 일관되게 끌어올려, 현재 데이터에서 회귀 성능에 실질적으로 기여하는 것은 `원료등급`이 아니라 파생변수 쪽임을 ablation test가 보여준다.

In [ ]:
final_df['원료등급'].value_counts(normalize=True) * 100

In [ ]:
plot_grade_distribution(final_df, FIGURES_DIR)